# Data Reduction — Sprint 2

Genera el dataset final del sprint: selección de variables para Marketing,
Operaciones y Experiencia del Cliente. No se eliminan filas, solo columnas.

## Duda abierta: variables para estudios futuros

No incluidas todavía, pendientes de decidir con el equipo para una posible
regresión ampliada: `bedrooms`, `bathrooms`, `beds`,
`neighbourhood_district`/`neighbourhood_name`. `room_type` ya está incluida.

## Estrategia acordada

- Un registro por `apartment_id` (ya resuelto en Cleaning).
- Se mantienen los nulos de reseñas y `price_€`.
- `cat_*` se detecta por prefijo, no por lista cerrada.
- `is_instant_bookable`/`has_availability` deben venir en versión numérica
  (1/0); si no existen, el notebook se detiene con error.
- Se conservan `availability_*` y `occupancy_*`/`occupancy_rate_*` "por si
  acaso", aunque este notebook no las usa en ningún cálculo.
- Entrada: `Scripts/transform_dataset.csv`. Salida:
  `Data/clean_dataset_06_07_2026.csv`.

## 1. Importación de librerías

In [ ]:
from pathlib import Path
import pandas as pd

## 2. Localización de la raíz del proyecto

Busca la carpeta `Equip_34` subiendo desde el directorio actual, sin guardar
rutas absolutas del ordenador.

In [ ]:
def encontrar_raiz_proyecto(nombre_carpeta="Equip_34"):
    """
    Busca la carpeta raíz del proyecto subiendo desde el directorio
    actual.
    """
    actual = Path.cwd()

    for carpeta in [actual] + list(actual.parents):
        if carpeta.name == nombre_carpeta:
            return carpeta

    raise FileNotFoundError(
        f"No se encontró la carpeta '{nombre_carpeta}'."
    )


raiz_proyecto = encontrar_raiz_proyecto()

ruta_entrada = (
    raiz_proyecto
    / "Scripts"
    / "transform_dataset.csv"
)

ruta_salida = (
    raiz_proyecto
    / "Data"
    / "clean_dataset_06_07_2026.csv"
)

print("Archivo de entrada:", f"Scripts/{ruta_entrada.name}")
print("Archivo de salida:", f"Data/{ruta_salida.name}")

## 3. Carga del dataset

Carga completa en memoria; solo se escribe al final, tras las
validaciones.

In [ ]:
df_original = pd.read_csv(ruta_entrada)

df = df_original.copy()

print("Dimensiones iniciales:", df.shape)
df.head()

## 4. Selección de variables necesarias

`cat_*` se detecta por prefijo. `is_instant_bookable`/`has_availability`
deben existir en su versión `_numeric`; si no, el notebook falla aquí.

In [ ]:
# Columnas fijas acordadas para este sprint
columnas_fijas = [
    # Identificación y segmentación
    "apartment_id",
    "city",
    "insert_date",
    "room_type",

    # Marketing
    "price_€",
    "accommodates",

    # Experiencia del cliente
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",

    # Operaciones — se mantienen "por si acaso"
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "occupancy_30",
    "occupancy_60",
    "occupancy_90",
    "occupancy_365",
    "occupancy_rate_30",
    "occupancy_rate_60",
    "occupancy_rate_90",
    "occupancy_rate_365",
]

# Columnas cat_* detectadas dinámicamente
columnas_cat = sorted(
    columna for columna in df.columns
    if columna.startswith("cat_")
)

if not columnas_cat:
    raise ValueError(
        "No se han encontrado columnas 'cat_*'. "
        "Data Transformation debe generarlas antes."
    )

print("Columnas cat_* detectadas:", len(columnas_cat))
print(columnas_cat)


def exigir_columna_numerica(nombre_base, df):
    """Exige la existencia de la versión numérica (1/0)."""
    version_numerica = f"{nombre_base}_numeric"
    if version_numerica not in df.columns:
        raise ValueError(
            f"No se ha encontrado '{version_numerica}'. "
            f"Se requiere la versión binaria, no "
            f"'{nombre_base}' (VERDADERO/FALSO)."
        )
    print(f"Usando versión binaria: '{version_numerica}'")
    return version_numerica


columna_instant_bookable = exigir_columna_numerica(
    "is_instant_bookable", df
)
columna_has_availability = exigir_columna_numerica(
    "has_availability", df
)

columnas_reducidas = (
    columnas_fijas
    + columnas_cat
    + [columna_instant_bookable, columna_has_availability]
)

columnas_faltantes = [
    columna
    for columna in columnas_reducidas
    if columna not in df.columns
]

if columnas_faltantes:
    raise ValueError(
        "Faltan columnas necesarias: "
        f"{columnas_faltantes}"
    )

print("Número de columnas seleccionadas:", len(columnas_reducidas))

## 5. Validaciones antes de reducir

Comprueba unicidad de `apartment_id`, rangos de puntuaciones y
disponibilidad, y que `cat_*` sea binaria. No repite Cleaning/Transformation,
solo verifica su resultado.

In [ ]:
duplicados_id = df["apartment_id"].duplicated().sum()

print("Apartment ID duplicados:", duplicados_id)

if duplicados_id > 0:
    raise ValueError(
        "El dataset contiene apartment_id duplicados. "
        "Debe revisarse Data Cleaning."
    )

rating_valido = (
    df["review_scores_rating"].isna()
    | df["review_scores_rating"].between(0, 100)
)

if not rating_valido.all():
    raise ValueError(
        "review_scores_rating fuera de 0-100. "
        "Verificar el reescalado de Transformation."
    )

subpuntuaciones = [
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
]

for columna in subpuntuaciones:
    valores_validos = (
        df[columna].isna()
        | df[columna].between(0, 10)
    )

    if not valores_validos.all():
        raise ValueError(
            f"{columna} contiene valores fuera de 0-10."
        )

precio_valido = (
    df["price_€"].isna()
    | (df["price_€"] > 0)
)

if not precio_valido.all():
    raise ValueError(
        "price_€ contiene valores nulos o negativos."
    )

for columna in columnas_cat:
    valores_unicos = set(df[columna].dropna().unique())
    if not valores_unicos.issubset({0, 1}):
        raise ValueError(
            f"La columna '{columna}' no es binaria (0/1)."
        )

limites_disponibilidad = {
    "availability_30": 30,
    "availability_60": 60,
    "availability_90": 90,
    "availability_365": 365,
}

for columna, limite in limites_disponibilidad.items():
    valores_validos = (
        df[columna].isna()
        | df[columna].between(0, limite)
    )

    if not valores_validos.all():
        raise ValueError(
            f"{columna} contiene valores fuera de "
            f"0-{limite}."
        )

limites_ocupacion = {
    "occupancy_30": 30,
    "occupancy_60": 60,
    "occupancy_90": 90,
    "occupancy_365": 365,
}

for columna, limite in limites_ocupacion.items():
    valores_validos = (
        df[columna].isna()
        | df[columna].between(0, limite)
    )

    if not valores_validos.all():
        raise ValueError(
            f"{columna} contiene valores fuera de "
            f"0-{limite}."
        )

columnas_tasa_ocupacion = [
    "occupancy_rate_30",
    "occupancy_rate_60",
    "occupancy_rate_90",
    "occupancy_rate_365",
]

for columna in columnas_tasa_ocupacion:
    valores_validos = (
        df[columna].isna()
        | df[columna].between(0, 100)
    )

    if not valores_validos.all():
        raise ValueError(
            f"{columna} contiene valores fuera de 0-100."
        )

print("Validaciones superadas.")

## 6. Aplicación de Data Reduction

In [ ]:
df_reduced = df[columnas_reducidas].copy()

print("Dimensiones originales:", df.shape)
print("Dimensiones finales:", df_reduced.shape)

reduccion_columnas_pct = round(
    (1 - df_reduced.shape[1] / df.shape[1]) * 100,
    2,
)

print(
    "Reducción del número de columnas:",
    f"{reduccion_columnas_pct}%",
)

df_reduced.head()

## 7. Cobertura por perfil

Diagnóstico de cuántos registros son utilizables por cada perfil, sin
eliminar filas del dataset final.

In [ ]:
cobertura = pd.DataFrame(
    {
        "perfil": [
            "Marketing",
            "Operaciones",
            "Experiencia del cliente",
        ],
        "registros_utilizables": [
            df_reduced[
                [
                    "city",
                    "room_type",
                    "price_€",
                ]
            ].dropna().shape[0],

            df_reduced[
                [
                    "city",
                    columna_instant_bookable,
                    "availability_30",
                    "availability_60",
                    "availability_90",
                    "availability_365",
                    "occupancy_30",
                    "occupancy_rate_30",
                ]
            ].dropna().shape[0],

            df_reduced[
                [
                    "city",
                    "review_scores_rating",
                ]
            ].dropna().shape[0],
        ],
    }
)

cobertura

## 8. Validación final

In [ ]:
if len(df_reduced) != len(df):
    raise ValueError(
        "Data Reduction ha modificado el número de filas."
    )

if not df_reduced["apartment_id"].is_unique:
    raise ValueError(
        "El dataset final contiene apartment_id duplicados."
    )

if list(df_reduced.columns) != columnas_reducidas:
    raise ValueError(
        "Las columnas finales no coinciden con la selección."
    )

print(
    "Filas conservadas:",
    len(df_reduced),
    "de",
    len(df),
)

print(
    "Apartment ID único:",
    df_reduced["apartment_id"].is_unique,
)

print(
    "Nulos en review_scores_rating:",
    df_reduced["review_scores_rating"].isna().sum(),
)

df_reduced.info()

## 9. Exportación del dataset final

Sobrescribe `clean_dataset_06_07_2026.csv` en `Data/`, que queda como
dataset final tras Cleaning + Transformation + Reduction.

In [ ]:
df_reduced.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8",
)

print(
    "Dataset final guardado correctamente en:",
    f"Data/{ruta_salida.name}",
)

## Resultado

Dataset final del Sprint 2: identificación, precio y capacidad, `cat_*`,
reserva instantánea y disponibilidad en binario, `availability_*`/
`occupancy_*` de reserva, y puntuaciones de experiencia.

**Pendiente de confirmar con el equipo:**
1. Variables extra para regresión futura (ver "Duda abierta").
2. Revisar que las `cat_*` detectadas son las 7 esperadas.
3. `is_instant_bookable`/`has_availability` exigen versión binaria; el
   notebook falla si no existe.